In [34]:
from torchvision import datasets, transforms as T, models
import numpy as np
from torchinfo import summary
import torch.nn as nn
import torch
from torch.utils.data import DataLoader
import torch.optim as optim

In [35]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [36]:
transform_train = T.Compose([
    T.RandomCrop(32, padding=4),
    T.RandomHorizontalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
transform_test = T.Compose([
    T.ToTensor(),
    T.Normalize([0.5071, 0.4867, 0.4408], [0.2675, 0.2565, 0.2761])
])
train_ds = datasets.CIFAR100("./data/", train=True, transform=transform_train, download=True)
test_ds = datasets.CIFAR100("./data/", train=False, transform=transform_test, download=True)
print(train_ds.data.shape, test_ds.data.shape)
print(len(np.unique(test_ds.classes)))

(50000, 32, 32, 3) (10000, 32, 32, 3)
100


In [37]:
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for param in model.parameters():
    param.requires_grad = False
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
model.fc = nn.Linear(model.fc.in_features, 100)
model = model.to(device)
summary(model)

Layer (type:depth-idx)                   Param #
ResNet                                   --
├─Conv2d: 1-1                            1,728
├─BatchNorm2d: 1-2                       (128)
├─ReLU: 1-3                              --
├─Identity: 1-4                          --
├─Sequential: 1-5                        --
│    └─BasicBlock: 2-1                   --
│    │    └─Conv2d: 3-1                  (36,864)
│    │    └─BatchNorm2d: 3-2             (128)
│    │    └─ReLU: 3-3                    --
│    │    └─Conv2d: 3-4                  (36,864)
│    │    └─BatchNorm2d: 3-5             (128)
│    └─BasicBlock: 2-2                   --
│    │    └─Conv2d: 3-6                  (36,864)
│    │    └─BatchNorm2d: 3-7             (128)
│    │    └─ReLU: 3-8                    --
│    │    └─Conv2d: 3-9                  (36,864)
│    │    └─BatchNorm2d: 3-10            (128)
├─Sequential: 1-6                        --
│    └─BasicBlock: 2-3                   --
│    │    └─Conv2d: 3-11     

In [38]:
import copy

def train(model, epochs, optimizer, train_loader, test_loader, criterion, device):
    best_accuracy = 0.0
    best_model_wts = copy.deepcopy(model.state_dict())  # guard: ensure always initialized
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for batch_X, batch_y in train_loader:
            batch_X, batch_y = batch_X.to(device, non_blocking=True), batch_y.to(device, non_blocking=True)
            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        train_loss = running_loss / len(train_loader)

        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0   
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                batch_X = batch_X.to(device, non_blocking=True)
                batch_y = batch_y.to(device, non_blocking=True)
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                val_loss += loss.item()

                _, predicted = torch.max(outputs.data, 1)
                total += batch_y.size(0)
                correct += (predicted == batch_y).sum().item()
        scheduler.step()
        val_loss = val_loss / len(test_loader) # Replace with len(val_loader)
        val_accuracy = correct / total
        
        # --- Save Best Model ---
        if val_accuracy > best_accuracy:
            best_accuracy = val_accuracy
            best_model_wts = copy.deepcopy(model.state_dict())
    
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch: {epoch+1}/{epochs}\tTrain Loss: {train_loss:.4f}\tVal Loss: {val_loss:.4f}\tVal Acc: {val_accuracy:.4f}\tLR: {lr:.2e}")    
        model.load_state_dict(best_model_wts)
    print(f"Training complete. Best Validation Accuracy: {best_accuracy:.4f}")

In [39]:
batch_size = 128
epochs = 10

In [40]:
train_loader = DataLoader(train_ds, batch_size, shuffle=True, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
test_loader = DataLoader(test_ds, batch_size, shuffle=False, num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=4)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

In [41]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
train(model, epochs, optimizer, train_loader, test_loader, criterion, device)

C:\Users\Manos\AppData\Local\Temp\ipykernel_33404\858771321.py:34: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch: 1/10	Train Loss: 4.0184	Val Loss: 3.4620	Val Acc: 0.2455	LR: 9.76e-04
Epoch: 2/10	Train Loss: 3.4116	Val Loss: 3.1112	Val Acc: 0.3303	LR: 9.05e-04
Epoch: 3/10	Train Loss: 3.1815	Val Loss: 2.9385	Val Acc: 0.3736	LR: 7.94e-04
Epoch: 4/10	Train Loss: 3.0428	Val Loss: 2.8521	Val Acc: 0.3979	LR: 6.55e-04
Epoch: 5/10	Train Loss: 2.9617	Val Loss: 2.8635	Val Acc: 0.3968	LR: 5.00e-04
Epoch: 6/10	Train Loss: 2.9641	Val Loss: 2.8243	Val Acc: 0.4073	LR: 3.45e-04
Epoch: 7/10	Train Loss: 2.8944	Val Loss: 2.7839	Val Acc: 0.4211	LR: 2.06e-04
Epoch: 8/10	Train Loss: 2.8645	Val Loss: 2.7421	Val Acc: 0.4233	LR: 9.55e-05
Epoch: 9/10	Train Loss: 2.8401	Val Loss: 2.7049	Val Acc: 0.4379	LR: 2.45e-05
Epoch: 10/10	Train Loss: 2.8022	Val Loss: 2.6917	Val Acc: 0.4445	LR: 0.00e+00
Training complete. Best Validation Accuracy: 0.4445


In [42]:
for param in model.parameters():
    param.requires_grad = True

In [43]:
batch_size = 128
epochs = 30

In [45]:
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-4)
train(model, epochs, optimizer, train_loader, test_loader, criterion, device)

Epoch: 1/30	Train Loss: 1.3697	Val Loss: 1.7793	Val Acc: 0.6919	LR: 9.76e-04
Epoch: 2/30	Train Loss: 1.3490	Val Loss: 1.7417	Val Acc: 0.7018	LR: 9.05e-04
Epoch: 3/30	Train Loss: 1.3314	Val Loss: 1.7330	Val Acc: 0.7062	LR: 7.94e-04
Epoch: 4/30	Train Loss: 1.3188	Val Loss: 1.7421	Val Acc: 0.7121	LR: 6.55e-04
Epoch: 5/30	Train Loss: 1.3071	Val Loss: 1.7369	Val Acc: 0.7099	LR: 5.00e-04
Epoch: 6/30	Train Loss: 1.3076	Val Loss: 1.7917	Val Acc: 0.6983	LR: 3.45e-04
Epoch: 7/30	Train Loss: 1.3074	Val Loss: 1.7747	Val Acc: 0.7035	LR: 2.06e-04
Epoch: 8/30	Train Loss: 1.3094	Val Loss: 1.7704	Val Acc: 0.6964	LR: 9.55e-05
Epoch: 9/30	Train Loss: 1.3014	Val Loss: 1.7311	Val Acc: 0.7093	LR: 2.45e-05
Epoch: 10/30	Train Loss: 1.3098	Val Loss: 1.7351	Val Acc: 0.7180	LR: 0.00e+00
Epoch: 11/30	Train Loss: 1.3037	Val Loss: 1.7104	Val Acc: 0.7134	LR: 2.45e-05
Epoch: 12/30	Train Loss: 1.3024	Val Loss: 1.7408	Val Acc: 0.7072	LR: 9.55e-05
Epoch: 13/30	Train Loss: 1.3020	Val Loss: 1.7718	Val Acc: 0.6992	LR: 2.06